In [0]:
silver_df = spark.table("loan_default_catalog.silver_schema.loan")
display(silver_df)

In [0]:
fill_dict = {
    "loan_amnt": 0,
    "funded_amnt": 0,
    "term": "unknown",
    "term_num": 0,
    "int_rate": 0,
    "installment": 0,
    "grade": "unknown",
    "sub_grade": "unknown",
    "emp_length_num": 0,
    "home_ownership": "unknown",
    "annual_inc": 0,
    "verification_status": "unknown",
    "issue_year": 0,
    "issue_month": 0,
    "loan_status": "unknown"
}

silver_df = silver_df.fillna(fill_dict)

In [0]:
additional_fill_dict = {
    "purpose": "unknown",
    "addr_state": "unknown",
    "dti": 0,
    "delinq_2yrs": 0,
    "fico_range_low": 0,
    "fico_range_high": 0,
    "fico_avg": 0,
    "inq_last_6mths": 0,
    "open_acc": 0,
    "total_acc": 0,
    "revol_bal": 0,
    "revol_util": 0,
    "acc_open_past_24mths": 0,
    "mort_acc": 0,
    "num_rev_accts": 0
}

# Merge with previous fill_dict
fill_dict.update(additional_fill_dict)
silver_df = silver_df.fillna(fill_dict)


In [0]:
more_numeric_fill_dict = {
    "open_acc": 0,
    "total_acc": 0,
    "revol_bal": 0,
    "revol_util": 0,
    "acc_open_past_24mths": 0,
    "mort_acc": 0,
    "num_rev_accts": 0,
    "num_actv_rev_tl": 0,
    "num_tl_30dpd": 0,
    "num_tl_90g_dpd_24m": 0,
    "pct_tl_nvr_dlq": 0,
    "pub_rec_bankruptcies": 0,
    "tax_liens": 0,
    "installment_to_income": 0,
    "credit_history_years": 0
}

fill_dict.update(more_numeric_fill_dict)
silver_df = silver_df.fillna(fill_dict)


In [0]:
import pyspark.sql.functions as F
null_counts = silver_df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in silver_df.columns]).collect()[0].asDict()
columns_with_nulls = [(col, count) for col, count in null_counts.items() if count > 0]
if columns_with_nulls:
    display(spark.createDataFrame(columns_with_nulls, ["column", "null_count"]))
else:
    print("Not any null present")

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE loan_default_catalog.gold_schema.loan_features_wide (
    id STRING,
    -- numeric
    loan_amnt DOUBLE,
    funded_amnt DOUBLE,
    term_num INT,
    int_rate DOUBLE,
    installment DOUBLE,
    annual_inc DOUBLE,
    dti DOUBLE,
    installment_to_income DOUBLE,
    emp_length_num INT,
    fico_avg DOUBLE,
    revol_bal DOUBLE,
    revol_util DOUBLE,
    pct_tl_nvr_dlq DOUBLE,
    
    issue_year INT,
    issue_month INT,
    credit_history_years INT,
    -- encoded
    grade_vec ARRAY<DOUBLE>,
    sub_grade_vec ARRAY<DOUBLE>,
    purpose_vec ARRAY<DOUBLE>,
    home_ownership_vec ARRAY<DOUBLE>,
    verification_status_vec ARRAY<DOUBLE>,
    addr_state_vec ARRAY<DOUBLE>,
    target_default INT,
    ingestion_ts TIMESTAMP
)
USING DELTA;

In [0]:
display(silver_df)

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

categorical_cols = [
    "grade","sub_grade","purpose",
    "home_ownership","verification_status",
    "addr_state"
]

indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=f"{c}_idx",
        handleInvalid="keep"
    )
    for c in categorical_cols
]

encoders = [
    OneHotEncoder(
        inputCol=f"{c}_idx",
        outputCol=f"{c}_vec"
    )
    for c in categorical_cols
]

numeric_cols = [
    "loan_amnt","funded_amnt","term_num",
    "int_rate","installment","annual_inc","dti",
    "installment_to_income","emp_length_num","fico_avg",
    "revol_bal","revol_util","pct_tl_nvr_dlq",
    "issue_year","issue_month","credit_history_years"
]

assembler = VectorAssembler(
    inputCols=numeric_cols + [f"{c}_vec" for c in categorical_cols],
    outputCol="features",
    handleInvalid="keep"
)

pipeline = Pipeline(stages=indexers + encoders + [assembler])


In [0]:
%sql
CREATE TABLE loan_default_catalog.gold_schema.loan_model_input (
    id STRING,
    features ARRAY<DOUBLE>,
    target_default INT,
    ingestion_ts TIMESTAMP
)
USING DELTA;

In [0]:
import gc

# delete anything model-like
try:
    del gold_pipeline_model
except:
    pass

try:
    del pipeline
except:
    pass

try:
    del model
except:
    pass

gc.collect()


In [0]:
gold_pipeline_model = pipeline.fit(silver_df)

In [0]:
gold_features_df = gold_pipeline_model.transform(silver_df)


In [0]:
from pyspark.sql.functions import current_timestamp

gold_final_df = gold_features_df.select(
    "id",
    "features",
    "target_default",
    current_timestamp().alias("ingestion_ts")
)


In [0]:
from pyspark.ml.functions import vector_to_array

gold_final_df = gold_final_df.withColumn(
    "features",
    vector_to_array("features")
)


In [0]:
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import current_timestamp

gold_storage_df = gold_features_df.select(
    "id",
    vector_to_array("features").alias("features"),
    "target_default",
    current_timestamp().alias("ingestion_ts")
)

gold_storage_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("loan_default_catalog.gold_schema.loan_model_input")


In [0]:
gold_df = spark.table("loan_default_catalog.gold_schema.loan_model_input")
display(gold_df)

In [0]:
train_df, test_df = gold_df.randomSplit([0.8, 0.2], seed=42)


In [0]:
import mlflow
import mlflow.spark

from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import col
from pyspark.ml.functions import array_to_vector
import os

# Set MLFLOW_DFS_TMP as environment variable
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/loan_default_catalog/gold_schema/ml_artifacts"

# Filter out rows with null features or label
train_df_clean = train_df.filter(col("features").isNotNull() & col("target_default").isNotNull())
train_df_clean = train_df_clean.withColumn("features", array_to_vector("features"))
train_df_clean = train_df_clean.filter(~col("features").isNull())

test_df_clean = test_df.filter(col("features").isNotNull() & col("target_default").isNotNull())
test_df_clean = test_df_clean.withColumn("features", array_to_vector("features"))
test_df_clean = test_df_clean.filter(~col("features").isNull())

display(train_df_clean.groupBy("target_default").count())
display(test_df_clean.groupBy("target_default").count())

with mlflow.start_run():

    lr = LogisticRegression(
        featuresCol="features",
        labelCol="target_default",
        maxIter=50
    )
    sample_input = test_df.select("features").limit(5).toPandas()
    model = lr.fit(train_df_clean)
    preds = model.transform(test_df_clean)
    sample_output = preds.select("prediction", "probability").limit(5).toPandas()
    from mlflow.models.signature import infer_signature

    signature = infer_signature(sample_input, sample_output)

    evaluator = BinaryClassificationEvaluator(
        labelCol="target_default",
        metricName="areaUnderROC"
    )

    auc = evaluator.evaluate(preds)

    mlflow.log_metric("auc", auc)

    mlflow.spark.log_model(
        model,
        artifact_path="model",
        signature=signature,
        input_example=sample_input
    )


In [0]:
with mlflow.start_run():
    run_id ="f83af3bb50d141ae9054258cbb08e65e" 
    print(run_id)


In [0]:


model_name = "loan_default_catalog.gold_schema.model"

model_uri = f"runs:/{run_id}/model"

mlflow.register_model(
    model_uri=model_uri,
    name=model_name
)


In [0]:
loan_default_catalog.gold_schema.sincere-crab-157


In [0]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

client.set_registered_model_alias(
    name=model_name,
    alias="Production",
    version=1
)


In [0]:
from mlflow.tracking import MlflowClient
client = MlflowClient()
model_name = "loan_default_catalog.gold_schema.model"
versions = [int(m.version) for m in client.search_model_versions(f"name='{model_name}'")]
if versions:
    latest_version = max(versions)
    print("Latest version:", latest_version)
else:
    print(f"No versions found for model: {model_name}")

In [0]:
import mlflow.spark

model_name = "loan_default_catalog.gold_schema.model"

prod_model_uri = f"models:/{model_name}@Production"

model = mlflow.spark.load_model(prod_model_uri)


In [0]:
gold_features_df = spark.table(
    "loan_default_catalog.gold_schema.loan_model_input"
)


In [0]:
display(gold_features_df)

In [0]:
model_name = "loan_default_catalog.gold_schema.model"
prod_model_uri = f"models:/{model_name}@Production"

model = mlflow.spark.load_model(prod_model_uri)


In [0]:
from pyspark.ml.functions import array_to_vector
gold_features_df = gold_features_df.withColumn("features", array_to_vector("features"))

In [0]:
scored_df = model.transform(gold_features_df)


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS loan_default_catalog.gold_schema.ml_artifacts;

In [0]:
final_scores = scored_df.select(
    "id",
    "prediction",
    "probability"
)


In [0]:
%sql
CREATE TABLE loan_default_catalog.gold_schema.loan_predictions (
    id STRING,
    prediction DOUBLE,
    probability ARRAY<DOUBLE>,
    scored_at TIMESTAMP
)
USING DELTA;

In [0]:
from pyspark.sql.functions import current_timestamp

final_scores = final_scores.withColumn("scored_at", current_timestamp())

final_scores.write.format("delta") \
    .mode("append") \
    .saveAsTable("loan_default_catalog.gold_schema.loan_predictions")
